In [1]:
from datasets import load_dataset
import os, zipfile
from pathlib import Path
from urllib import request


d:\Text-Summarizer\textS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%pwd

'd:\\Text-Summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Text-Summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL : str
    local_data_file : Path
    unzip_dir : Path

    

In [6]:
from textSummarizer.constants import *

from textSummarizer.utils.common import read_yaml , create_directories


In [7]:
from textSummarizer.constants.__init__ import CONFIG_FILE_PATH, PARAMS_FILE_PATH
print(CONFIG_FILE_PATH)
print(PARAMS_FILE_PATH)


config\config.yaml
params.yaml


In [8]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir= config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config


In [9]:
import os 
import urllib.request as request
import zipfile
from textSummarizer.logging import logger


In [10]:
import importlib
import textSummarizer.utils.common as common
importlib.reload(common)
from textSummarizer.utils.common import get_size


In [11]:
class DataIngestion:
    def __init__(self , config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        # if not os.path.exists(self.config.local_data_file):
        #     filename , headers = request.urlretrieve(
        #         url = self.config.source_URL,
        #         filename=self.config.local_data_file
        #     )
        #     logger.info(f"{filename} download! with following info: \n{headers}")
        # else:
        #     logger.info(f"FIle already exists of size : {get_size(Path(self.config.local_data_file))}")

        # Case 1: Hugging Face dataset name instead of URL
        if self.config.source_URL.startswith("hf://"):
            dataset_name = self.config.source_URL.replace("hf://", "")
            ds = load_dataset(dataset_name)

            os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)

            # Save dataset splits into a zip file
            with zipfile.ZipFile(self.config.local_data_file, 'w') as zipf:
                for split in ds.keys():
                    filename = f"{split}.json"
                    path = os.path.join(os.path.dirname(self.config.local_data_file), filename)
                    ds[split].to_json(path)
                    zipf.write(path, arcname=filename)

        # Case 2: Normal URL download
        else:
            if not os.path.exists(self.config.local_data_file):
                filename, headers = request.urlretrieve(
                    url=self.config.source_URL,
                    filename=self.config.local_data_file
                )
                logger.info(f"{filename} downloaded with info: \n{headers}")
            else:
                logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """
        zip_file_path:str
        Extracts the zip file into the data directory
        Funtion returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [12]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-07 10:56:17,554:INFO:common:yaml file: config\config.yaml loaded successfully]
[2026-10-07 10:56:17,556:INFO:common:yaml file: params.yaml loaded successfully]
[2026-10-07 10:56:17,558:INFO:common:created directory at: artifacts]
[2026-10-07 10:56:17,559:INFO:common:created directory at: artifacts/data_ingestion]
[2026-10-07 10:56:17,984:INFO:_client:HTTP Request: HEAD https://huggingface.co/datasets/knkarthick/samsum/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"]


[2026-10-07 10:56:17,985:WARNING:_http:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-10-07 10:56:18,022:INFO:_client:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/knkarthick/samsum/6b929ff10edec703164e3ddb2e94aae058c9ab5f/README.md?%2Fdatasets%2Fknkarthick%2Fsamsum%2Fresolve%2Fmain%2FREADME.md=&etag=%2269f653ecfa222eb727bd79fdcdbb8248cf6e7298%22 "HTTP/1.1 200 OK"]
[2026-10-07 10:56:18,281:INFO:_client:HTTP Request: HEAD https://huggingface.co/datasets/knkarthick/samsum/resolve/6b929ff10edec703164e3ddb2e94aae058c9ab5f/samsum.py "HTTP/1.1 404 Not Found"]
[2026-10-07 10:56:19,312:INFO:_client:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/knkarthick/samsum/knkarthick/samsum.py "HTTP/1.1 404 Not Found"]
[2026-10-07 10:56:19,571:INFO:_client:HTTP Request: GET https://huggingface.co/api/datasets/knkarthick/samsum/revision/6b929ff10ed

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 79.01ba/s]


In [14]:
from datasets import load_dataset

ds = load_dataset("json", data_files="artifacts/data_ingestion/train.json")
print(ds["train"][0])



[2026-10-07 11:02:38,120:INFO:_client:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"]


Generating train split: 14731 examples [00:00, 98361.54 examples/s] 

{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}
